# TTM-R3 BTCUSDT 30m logReturn — Multivariate Fine-Tuning + Walk-Forward Evaluation

**Model**: [ibm-research/ttm-r3](https://huggingface.co/ibm-research/ttm-r3)  
**Repo**: [zaynety7/granite-tsfm](https://github.com/zaynety7/granite-tsfm)  

### Key TTM-R3 features used
- `mode="mix_channel"` + `forecast_channel_mixing=True` — cross-channel multivariate decoding
- `conditional_columns` — 11 selected Z-score features passed as auxiliary channels
- `prediction_filter_length` — trim native prediction_length output to 6 steps (30 min) at inference
- Quantile head (`loss="pinball"`, quantiles [0.05,0.1,0.25,0.5,0.75,0.9,0.95])
- Walk-forward in-sample / out-of-sample split with rolling re-evaluation

### Evaluation framework (from tradeTest.md)
- **Model layer**: Pinball Loss per quantile, PICP (80%/90% PI), Quantile Crossing rate, Calibration
- **Strategy layer**: Sharpe, Sortino, Max Drawdown, Hit Rate × Profit Factor, Signal Coverage

In [ ]:
# ── Cell 1 : Install dependencies ──────────────────────────────────────────
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

# Install granite-tsfm from the fork (editable so imports resolve)
pip('git+https://github.com/zaynety7/granite-tsfm.git')
pip('transformers>=4.40.0', 'datasets', 'accelerate', 'einops')
pip('python-binance', 'requests')   # data download helpers
pip('plotly', 'kaleido')             # charts

In [ ]:
# ── Cell 2 : Imports ───────────────────────────────────────────────────────
import os, warnings, gc
import numpy as np
import pandas as pd
from pathlib import Path

import torch
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback

from tsfm_public import TinyTimeMixerForPrediction, TimeSeriesPreprocessor
from tsfm_public.toolkit.dataset import ForecastDFDataset
from tsfm_public.toolkit.util import select_by_index

warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 1  Data Download & Feature Engineering

In [ ]:
# ── Cell 4 : Download 5-minute Binance klines via public REST ─────────────
# Covers 2024-02-17 → 2026-04-24 for BTCUSDT
# Sources: futures (cm), spot (sp), premium index (prem)

import requests, time
from datetime import datetime, timezone

BASE_FAPI = 'https://fapi.binance.com'
BASE_API  = 'https://api.binance.com'

def fetch_klines(base_url: str, endpoint: str, symbol: str,
                 start_ms: int, end_ms: int, interval: str = '5m') -> pd.DataFrame:
    """Paginate through Binance kline API (1000 bars per request)."""
    records = []
    url = f'{base_url}{endpoint}'
    cur = start_ms
    while cur < end_ms:
        resp = requests.get(url, params=dict(
            symbol=symbol, interval=interval,
            startTime=cur, endTime=end_ms, limit=1500), timeout=30)
        resp.raise_for_status()
        data = resp.json()
        if not data:
            break
        records.extend(data)
        cur = int(data[-1][0]) + 1
        time.sleep(0.12)
    cols = ['open_time','open','high','low','close','volume',
            'close_time','quote_volume','trades','taker_buy_base',
            'taker_buy_quote','ignore']
    df = pd.DataFrame(records, columns=cols)
    df['timestamp'] = pd.to_datetime(df['open_time'], unit='ms', utc=True)
    for c in ['open','high','low','close','volume','quote_volume',
              'taker_buy_base','taker_buy_quote']:
        df[c] = df[c].astype(float)
    return df.drop_duplicates('timestamp').sort_values('timestamp').reset_index(drop=True)

# ── date range ────────────────────────────────────────────────────────────
START_MS = int(datetime(2024, 2, 17, tzinfo=timezone.utc).timestamp() * 1000)
END_MS   = int(datetime(2026, 4, 24, 23, 59, tzinfo=timezone.utc).timestamp() * 1000)

print('Downloading futures klines...')
df_cm = fetch_klines(BASE_FAPI, '/fapi/v1/klines', 'BTCUSDT', START_MS, END_MS)

print('Downloading spot klines...')
df_sp = fetch_klines(BASE_API, '/api/v3/klines', 'BTCUSDT', START_MS, END_MS)

print('Downloading premium index klines...')
df_prem = fetch_klines(BASE_FAPI, '/fapi/v1/premiumIndexKlines', 'BTCUSDT', START_MS, END_MS)

print(f'Futures bars: {len(df_cm):,}  |  Spot bars: {len(df_sp):,}  |  Prem bars: {len(df_prem):,}')

In [ ]:
# ── Cell 5 : Feature Engineering (5-min level → aggregate to 30-min) ──────
# VWAP formula: quote_volume (U-denominated) / volume (coin-denominated)

def add_vwap(df: pd.DataFrame, prefix: str) -> pd.DataFrame:
    df = df.copy()
    # VWAP = Σ(quote_volume) / Σ(volume)  — optimised U-denominated formula
    df[f'{prefix}_vwap'] = df['quote_volume'] / df['volume']
    return df

def rolling_z(s: pd.Series, window: int) -> pd.Series:
    mu  = s.rolling(window, min_periods=1).mean()
    sig = s.rolling(window, min_periods=1).std().replace(0, np.nan)
    return (s - mu) / sig

# ── merge on timestamp ────────────────────────────────────────────────────
df = (df_cm[['timestamp','open','high','low','close','volume','quote_volume',
              'taker_buy_base','taker_buy_quote']]
       .rename(columns={c: f'cm_{c}' for c in ['open','high','low','close',
                                                  'volume','quote_volume',
                                                  'taker_buy_base','taker_buy_quote']})
       .merge(df_sp[['timestamp','open','high','low','close','volume',
                      'quote_volume','taker_buy_base','taker_buy_quote']]
              .rename(columns={c: f'sp_{c}' for c in ['open','high','low','close',
                                                         'volume','quote_volume',
                                                         'taker_buy_base','taker_buy_quote']}),
              on='timestamp', how='inner')
       .merge(df_prem[['timestamp','close']].rename(columns={'close':'premium_close'}),
              on='timestamp', how='inner'))

# ── VWAP (U-denominated / coin-denominated) ───────────────────────────────
df['cm_vwap'] = df['cm_quote_volume'] / df['cm_volume']
df['sp_vwap'] = df['sp_quote_volume'] / df['sp_volume']

# ── CVD (Cumulative Volume Delta) ─────────────────────────────────────────
df['cm_cvd_delta'] = df['cm_taker_buy_base'] - (df['cm_volume'] - df['cm_taker_buy_base'])
df['sp_cvd_delta'] = df['sp_taker_buy_base'] - (df['sp_volume'] - df['sp_taker_buy_base'])
df['cm_cvd_cum']   = df['cm_cvd_delta'].cumsum()
df['sp_cvd_cum']   = df['sp_cvd_delta'].cumsum()

# ── Basis ─────────────────────────────────────────────────────────────────
df['basis_cm_spot']     = (df['cm_close'] - df['sp_close']) / df['sp_close']
df['basis_compression'] = df['basis_cm_spot'].diff()  # 1-bar change
df['basis_from_trades'] = (df['cm_vwap'] - df['sp_vwap']) / df['sp_vwap'] * 10_000  # bps

# ── CVD derived ──────────────────────────────────────────────────────────
df['cm_cvd_change_rate']    = df['cm_cvd_cum'].pct_change().replace([np.inf, -np.inf], np.nan)
df['sp_cvd_change_rate']    = df['sp_cvd_cum'].pct_change().replace([np.inf, -np.inf], np.nan)
df['cm_cvd_acceleration']   = df['cm_cvd_change_rate'].diff()
df['sp_cvd_acceleration']   = df['sp_cvd_change_rate'].diff()
df['cvd_diff_cm_spot']      = df['cm_cvd_cum'] - df['sp_cvd_cum']

# ── VWAP distance from close ──────────────────────────────────────────────
df['cm_vwap_dist_cm'] = (df['cm_close'] - df['cm_vwap']) / df['cm_vwap']

# ── Resample to 30-min ────────────────────────────────────────────────────
df = df.set_index('timestamp')

agg_rules = {
    'cm_close':              'last',
    'sp_close':              'last',
    'premium_close':         'last',
    'basis_cm_spot':         'last',
    'basis_compression':     'sum',
    'basis_from_trades':     'last',
    'cm_cvd_cum':            'last',
    'sp_cvd_cum':            'last',
    'cm_cvd_change_rate':    'last',
    'sp_cvd_change_rate':    'last',
    'cm_cvd_acceleration':   'last',
    'sp_cvd_acceleration':   'last',
    'cvd_diff_cm_spot':      'last',
    'cm_vwap_dist_cm':       'last',
}
df30 = df.resample('30min', label='right', closed='right').agg(agg_rules).dropna()
df30 = df30.reset_index()

# ── log return of futures close ───────────────────────────────────────────
df30['cm_logret'] = np.log(df30['cm_close'] / df30['cm_close'].shift(1))
df30 = df30.dropna().reset_index(drop=True)

# ── Z-score features (windows match selected_features_30m_with_formula.csv) 
z_specs = {
    'basis_cm_spot_z3':         ('basis_cm_spot',       3),
    'basis_compression_z24':    ('basis_compression',   24),
    'basis_from_trades_z3':     ('basis_from_trades',   3),
    'cm_cvd_acceleration_z24':  ('cm_cvd_acceleration', 24),
    'cm_cvd_change_rate_z3':    ('cm_cvd_change_rate',  3),
    'cm_cvd_cum_z3':            ('cm_cvd_cum',          3),
    'cm_vwap_dist_cm_z24':      ('cm_vwap_dist_cm',     24),
    'cvd_diff_cm_spot_z3':      ('cvd_diff_cm_spot',    3),
    'premium_close_z3':         ('premium_close',       3),
    'spot_cvd_acceleration_z12':('sp_cvd_acceleration', 12),
    'spot_cvd_change_rate_z3':  ('sp_cvd_change_rate',  3),
    'spot_cvd_cum_z3':          ('sp_cvd_cum',          3),
}
for feat, (src, w) in z_specs.items():
    df30[feat] = rolling_z(df30[src], w)

df30 = df30.dropna().reset_index(drop=True)
print(f'30-min bars: {len(df30):,}  |  columns: {list(df30.columns)}')

## 2  Column Schema & Walk-Forward Split

In [ ]:
# ── Cell 7 : Define column schema ─────────────────────────────────────────
TIMESTAMP_COL = 'timestamp'
TARGET_COL    = 'cm_logret'          # primary forecast target

# 11 conditional / auxiliary feature columns (Z-score normalised)
COND_COLS = [
    'basis_cm_spot_z3',
    'basis_compression_z24',
    'basis_from_trades_z3',
    'cm_cvd_acceleration_z24',
    'cm_cvd_change_rate_z3',
    'cm_cvd_cum_z3',
    'cm_vwap_dist_cm_z24',
    'cvd_diff_cm_spot_z3',
    'premium_close_z3',
    'spot_cvd_acceleration_z12',
    'spot_cvd_change_rate_z3',
    'spot_cvd_cum_z3',
]

# All observable input columns (target + conditional)
INPUT_COLS = [TARGET_COL] + COND_COLS

# ── TTM-R3 context / prediction lengths ──────────────────────────────────
# TTM-R3 native context window choices: 512 or 1024
# 30-min bars: 512 bars ≈ 10.7 days context  ✓
CONTEXT_LENGTH        = 512
# TTM-R3 native prediction length must match the pretrained checkpoint.
# ibm-research/ttm-r3 (512-96): prediction_length=96
# We use prediction_filter_length=6 at inference to get 6×30min = 3h ahead.
PREDICTION_LENGTH     = 96   # FIXED — do not change for fine-tuning
PREDICTION_FILTER_LEN = 6    # steps to evaluate (6 × 30min = 3h)

# ── Walk-forward split ────────────────────────────────────────────────────
# In-sample: 2024-02-17 → 2025-09-30  (~18 months for fine-tuning)
# Out-of-sample: 2025-10-01 → 2026-04-24  (~7 months evaluation)
SPLIT_DATE = pd.Timestamp('2025-10-01', tz='UTC')

df30['timestamp'] = pd.to_datetime(df30['timestamp'], utc=True)

train_df = df30[df30['timestamp'] < SPLIT_DATE].reset_index(drop=True)
test_df  = df30[df30['timestamp'] >= SPLIT_DATE].reset_index(drop=True)

# 10% of train → validation
val_split   = int(len(train_df) * 0.9)
val_df      = train_df.iloc[val_split:].reset_index(drop=True)
train_df    = train_df.iloc[:val_split].reset_index(drop=True)

print(f'Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}')

## 3  TimeSeriesPreprocessor

In [ ]:
# ── Cell 9 : Fit TimeSeriesPreprocessor ───────────────────────────────────
# Scaler: standard (zero-mean / unit-variance) per channel
# id_columns=[]: single-asset dataset

tsp = TimeSeriesPreprocessor(
    timestamp_column=TIMESTAMP_COL,
    target_columns=INPUT_COLS,       # all channels fed to TTM
    context_length=CONTEXT_LENGTH,
    prediction_length=PREDICTION_LENGTH,
    scaling=True,
    scaler_type='standard',
    encode_categorical=False,
)

train_dataset = tsp.get_dataset(
    train_df, split='train', fewshot_fraction=1.0)
val_dataset   = tsp.get_dataset(
    val_df, split='valid')
test_dataset  = tsp.get_dataset(
    test_df, split='test')

print('Datasets created.')
print(f'  train samples: {len(train_dataset):,}')
print(f'  val   samples: {len(val_dataset):,}')
print(f'  test  samples: {len(test_dataset):,}')

## 4  Load TTM-R3 & Configure Mix-Channel Multivariate Head

In [ ]:
# ── Cell 11 : Load TTM-R3 checkpoint & attach multivariate forecast head ──
# TTM-R3 new features leveraged:
#  1. mode='mix_channel'          — cross-channel Mixer in decoder
#  2. forecast_channel_mixing=True — decoder mixes all N channels
#  3. conditional_columns          — auxiliary channels are conditioned on
#     but filtered out of loss computation
#  4. prediction_filter_length     — only first K steps used at inference
#  5. loss='pinball', quantiles     — distributional probabilistic output

N_CHANNELS = len(INPUT_COLS)   # 1 target + 12 conditional = 13

model = TinyTimeMixerForPrediction.from_pretrained(
    'ibm-research/ttm-r3',
    # ── multivariate cross-channel settings ────────────────────────────
    num_input_channels=N_CHANNELS,
    mode='mix_channel',               # TTM-R3: cross-channel mixing in decoder
    forecast_channel_mixing=True,     # TTM-R3: enable channel mixing in forecast head
    # ── conditional columns (indices into INPUT_COLS) ──────────────────
    # conditional_columns are auxiliary — excluded from primary loss
    conditional_columns=list(range(1, N_CHANNELS)),  # indices 1..12
    # ── prediction length & filter ─────────────────────────────────────
    prediction_length=PREDICTION_LENGTH,             # keep at 96 (pretrained)
    prediction_filter_length=PREDICTION_FILTER_LEN,  # evaluate only first 6 steps
    # ── probabilistic quantile head ────────────────────────────────────
    loss='pinball',
    quantiles=[0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95],
    # ── freeze backbone, train heads only (initial phase) ──────────────
    head_dropout=0.1,
)

# Freeze backbone encoder, only train new head
for name, param in model.named_parameters():
    if 'head' not in name:
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,}  ({100*trainable/total:.1f}%)')

## 5  Fine-Tuning (Head-Only → Full Unfreeze)

In [ ]:
# ── Cell 13 : Phase 1 — Head-only fine-tuning ─────────────────────────────
OUT_DIR = Path('/kaggle/working/ttm_r3_btcusdt')
OUT_DIR.mkdir(parents=True, exist_ok=True)

training_args_p1 = TrainingArguments(
    output_dir=str(OUT_DIR / 'phase1'),
    overwrite_output_dir=True,
    num_train_epochs=20,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,
    learning_rate=1e-3,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    fp16=(DEVICE == 'cuda'),
    dataloader_num_workers=2,
    report_to='none',
    logging_steps=50,
)

trainer_p1 = Trainer(
    model=model,
    args=training_args_p1,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)
trainer_p1.train()
print('Phase 1 complete.')

In [ ]:
# ── Cell 14 : Phase 2 — Full model unfreeze fine-tuning ───────────────────
for param in model.parameters():
    param.requires_grad = True

training_args_p2 = TrainingArguments(
    output_dir=str(OUT_DIR / 'phase2'),
    overwrite_output_dir=True,
    num_train_epochs=30,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=128,
    learning_rate=5e-5,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    fp16=(DEVICE == 'cuda'),
    dataloader_num_workers=2,
    report_to='none',
    logging_steps=50,
)

trainer_p2 = Trainer(
    model=model,
    args=training_args_p2,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=7)],
)
trainer_p2.train()
print('Phase 2 complete. Model ready for inference.')

## 6  Walk-Forward Prediction

In [ ]:
# ── Cell 16 : Run inference on test set ───────────────────────────────────
# prediction_filter_length=6 clips output to first 6 horizon steps
# Only TARGET_COL (index 0) is evaluated; conditional cols are discarded.

from torch.utils.data import DataLoader

model.eval()
model.to(DEVICE)

loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

all_preds  = []  # (N, PREDICTION_FILTER_LEN, n_quantiles)
all_actual = []  # (N, PREDICTION_FILTER_LEN)

QUANTILES = [0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]

with torch.no_grad():
    for batch in loader:
        past_values = batch['past_values'].to(DEVICE)
        future_values = batch.get('future_values')

        outputs = model(past_values=past_values)
        # outputs.prediction_outputs: (B, prediction_filter_length, n_channels * n_quantiles)
        # or (B, prediction_filter_length, n_quantiles) when filtered to target
        preds = outputs.prediction_outputs  # shape depends on TTM-R3 version

        # Extract target channel (index 0) predictions
        # TTM-R3 with conditional_columns returns shape (B, filter_len, n_quantiles)
        # for the primary target channel after filtering
        if preds.dim() == 4:
            # (B, filter_len, n_channels, n_quantiles) → take channel 0
            preds = preds[:, :, 0, :]

        all_preds.append(preds.cpu().numpy())

        if future_values is not None:
            # future_values: (B, prediction_length, n_channels) → channel 0, first 6 steps
            fv = future_values[:, :PREDICTION_FILTER_LEN, 0].cpu().numpy()
            all_actual.append(fv)

preds_arr  = np.concatenate(all_preds,  axis=0)  # (N, 6, 7)
actual_arr = np.concatenate(all_actual, axis=0)  # (N, 6)

# Inverse-transform predictions (undo StandardScaler on target channel)
target_scaler = tsp.target_scaler  # ScalerWrapper

def inv_scale_target(arr_2d):
    """arr_2d: (N, K) — inverse-scale using target channel scaler."""
    mu  = target_scaler.mean_[0]
    sig = target_scaler.scale_[0]
    return arr_2d * sig + mu

# preds_arr: (N, 6, 7) → inv-scale each quantile slice
preds_inv  = np.stack([inv_scale_target(preds_arr[:, :, q]) for q in range(len(QUANTILES))], axis=-1)
actual_inv = inv_scale_target(actual_arr)

print(f'preds_inv shape : {preds_inv.shape}')
print(f'actual_inv shape: {actual_inv.shape}')

## 7  Evaluation — Model Layer (Statistical)

In [ ]:
# ── Cell 18 : Model-layer statistical evaluation ──────────────────────────
# Metrics: Pinball Loss, PICP (80%/90%), Quantile Crossing, Calibration

def pinball(y_true, y_pred, alpha):
    e = y_true - y_pred
    return np.mean(np.maximum(alpha * e, (alpha - 1) * e))

def picp(y_true, lower, upper):
    """Prediction Interval Coverage Probability."""
    return np.mean((y_true >= lower) & (y_true <= upper))

def quantile_crossing_rate(preds_nkq):
    """Fraction of (sample, step) pairs where any adjacent quantiles cross."""
    N, K, Q = preds_nkq.shape
    crossings = 0
    for q in range(Q - 1):
        crossings += np.sum(preds_nkq[:, :, q] > preds_nkq[:, :, q + 1])
    return crossings / (N * K * (Q - 1))

QIDX = {q: i for i, q in enumerate(QUANTILES)}

# ── per-step Pinball Loss ─────────────────────────────────────────────────
pb_results = {}
for step in range(PREDICTION_FILTER_LEN):
    row = {}
    for alpha, i in QIDX.items():
        row[f'PB_q{int(alpha*100):02d}'] = pinball(actual_inv[:, step],
                                                    preds_inv[:, step, i], alpha)
    pb_results[f'Step{step+1}'] = row

df_pb = pd.DataFrame(pb_results).T
print('=== Pinball Loss per step ===')
print(df_pb.round(6))

# ── PICP ─────────────────────────────────────────────────────────────────
for step in range(PREDICTION_FILTER_LEN):
    p80 = picp(actual_inv[:, step],
               preds_inv[:, step, QIDX[0.10]],
               preds_inv[:, step, QIDX[0.90]])
    p90 = picp(actual_inv[:, step],
               preds_inv[:, step, QIDX[0.05]],
               preds_inv[:, step, QIDX[0.95]])
    w80 = preds_inv[:, step, QIDX[0.90]] - preds_inv[:, step, QIDX[0.10]]
    print(f'Step {step+1}  PICP_80%={p80:.3f}  PICP_90%={p90:.3f}  '          f'Width_80%={np.mean(w80)*100:.4f}%')

# ── Quantile Crossing Rate ────────────────────────────────────────────────
qcr = quantile_crossing_rate(preds_inv)
print(f'\nQuantile Crossing Rate: {qcr:.4f}')

# ── Calibration (q50) ────────────────────────────────────────────────────
calib_50 = np.mean(actual_inv < preds_inv[:, :, QIDX[0.50]])
print(f'Calib_0.5 (fraction actual < q50): {calib_50:.4f}  (target=0.5000)')

## 8  Evaluation — Strategy Layer (Trading Metrics)

In [ ]:
# ── Cell 20 : Strategy-layer evaluation ───────────────────────────────────
# Signal rule (from tradeTest.md):
#   Long  : q10 > 0         (even pessimistic scenario is bullish)
#   Short : q90 < 0         (even optimistic scenario is bearish)
#   Flat  : q10 <= 0 <= q90 (distribution straddles zero)
#
# Position sizing: fractional Kelly via quantile VaR
#   mu_hat  = q50 (predicted median)
#   VaR90   = |q10| (downside 90% VaR for long)
#   f*      = (mu_hat / VaR90) * risk_budget   clipped to [-1, 1]

RISK_BUDGET = 0.25
STEP        = 0      # evaluate on first 30-min horizon step

q10 = preds_inv[:, STEP, QIDX[0.10]]
q50 = preds_inv[:, STEP, QIDX[0.50]]
q90 = preds_inv[:, STEP, QIDX[0.90]]
y   = actual_inv[:, STEP]

# Signal generation
long_sig  = (q10 > 0).astype(float)
short_sig = (q90 < 0).astype(float)
signal    = long_sig - short_sig  # {-1, 0, +1}

# Fractional Kelly sizing
var90  = np.abs(q10) + 1e-9
f_star = np.clip((q50 / var90) * RISK_BUDGET, -1.0, 1.0)
pos    = signal * np.abs(f_star)

pnl = pos * y  # period return of position

# ── Sharpe / Sortino ─────────────────────────────────────────────────────
def sharpe(r, ann=17520):   # 17520 × 30min = 1 year
    return np.mean(r) / (np.std(r) + 1e-9) * np.sqrt(ann)

def sortino(r, ann=17520):
    downside = r[r < 0]
    return np.mean(r) / (np.std(downside) + 1e-9) * np.sqrt(ann)

# ── Max Drawdown ─────────────────────────────────────────────────────────
cum_ret = np.cumprod(1 + pnl)
running_max = np.maximum.accumulate(cum_ret)
drawdown    = (cum_ret - running_max) / running_max
max_dd      = drawdown.min()

# ── Hit Rate / Profit Factor ──────────────────────────────────────────────
active = (signal != 0)
pnl_active = pnl[active]
hit_rate   = np.mean(pnl_active > 0)
gross_win  = pnl_active[pnl_active > 0].sum()
gross_loss = np.abs(pnl_active[pnl_active < 0].sum()) + 1e-9
profit_factor = gross_win / gross_loss
signal_coverage = active.mean()

print('=== Strategy Layer Metrics ===')
print(f'  Sharpe Ratio     : {sharpe(pnl):.3f}')
print(f'  Sortino Ratio    : {sortino(pnl):.3f}')
print(f'  Max Drawdown     : {max_dd*100:.2f}%')
print(f'  Hit Rate         : {hit_rate:.3f}')
print(f'  Profit Factor    : {profit_factor:.3f}')
print(f'  Signal Coverage  : {signal_coverage:.3f}')

## 9  Dual-Layer Diagnostic & Visualisation

In [ ]:
# ── Cell 22 : Dual-layer diagnostic (tradeTest.md logic) ─────────────────
print('=== Dual-Layer Diagnostic ===')
picp_80 = picp(actual_inv[:, 0],
               preds_inv[:, 0, QIDX[0.10]],
               preds_inv[:, 0, QIDX[0.90]])
_sr  = sharpe(pnl)
_cal = float(calib_50)

if picp_80 >= 0.78 and _sr < 0.5:
    print('→ PICP good but Sharpe low: signal direction correct, entry timing issue.')
elif picp_80 < 0.78 and _sr < 0.5:
    print('→ PICP low + Sharpe low: tail risk underestimated. Widen stop-loss 15-20%.')
elif picp_80 >= 0.78 and _sr >= 0.5:
    print('→ PICP good + Sharpe acceptable: model is working well.')
else:
    print('→ PICP low but Sharpe OK: threshold design may be compensating for poor calibration.')

if abs(_cal - 0.5) < 0.02:
    print(f'  Calib_0.5={_cal:.4f}: q50 highly reliable as trend midpoint.')
else:
    print(f'  Calib_0.5={_cal:.4f}: q50 biased, recalibrate threshold.')

In [ ]:
# ── Cell 23 : Visualisation — fan chart for first 500 test windows ────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots

N_PLOT = min(500, len(actual_inv))
steps  = np.arange(1, PREDICTION_FILTER_LEN + 1)

# Mean over first N_PLOT windows for each horizon step
mean_actual = actual_inv[:N_PLOT].mean(axis=0)
mean_q10    = preds_inv[:N_PLOT, :, QIDX[0.10]].mean(axis=0)
mean_q25    = preds_inv[:N_PLOT, :, QIDX[0.25]].mean(axis=0)
mean_q50    = preds_inv[:N_PLOT, :, QIDX[0.50]].mean(axis=0)
mean_q75    = preds_inv[:N_PLOT, :, QIDX[0.75]].mean(axis=0)
mean_q90    = preds_inv[:N_PLOT, :, QIDX[0.90]].mean(axis=0)

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.concatenate([steps, steps[::-1]]),
                          y=np.concatenate([mean_q90, mean_q10[::-1]]),
                          fill='toself', fillcolor='rgba(30,144,255,0.15)',
                          line=dict(color='rgba(0,0,0,0)'), name='80% PI'))
fig.add_trace(go.Scatter(x=np.concatenate([steps, steps[::-1]]),
                          y=np.concatenate([mean_q75, mean_q25[::-1]]),
                          fill='toself', fillcolor='rgba(30,144,255,0.30)',
                          line=dict(color='rgba(0,0,0,0)'), name='50% PI'))
fig.add_trace(go.Scatter(x=steps, y=mean_q50, mode='lines',
                          line=dict(color='royalblue', width=2), name='q50'))
fig.add_trace(go.Scatter(x=steps, y=mean_actual, mode='lines+markers',
                          line=dict(color='orange', width=2, dash='dot'),
                          name='Actual'))

fig.update_layout(
    title='TTM-R3  BTCUSDT 30m logReturn — Mean Quantile Fan (OOS)',
    xaxis_title='Horizon Step (× 30 min)',
    yaxis_title='log Return',
    template='plotly_dark',
)
fig.write_image(str(OUT_DIR / 'quantile_fan.png'), width=1000, height=500)
fig.show()
print('Chart saved.')

In [ ]:
# ── Cell 24 : Save predictions & diagnostics CSV ──────────────────────────
# Build per-window result table for Step 1 (first 30-min horizon)

test_timestamps = test_df['timestamp'].iloc[CONTEXT_LENGTH:].reset_index(drop=True)

result_rows = []
for i in range(len(actual_inv)):
    row = {'timestamp': test_timestamps.iloc[i] if i < len(test_timestamps) else np.nan,
           'actual':    actual_inv[i, 0]}
    for alpha, q in QIDX.items():
        row[f'q{int(alpha*100):02d}'] = preds_inv[i, 0, q]
    row['signal']   = signal[i]
    row['position'] = pos[i]
    row['pnl']      = pnl[i]
    result_rows.append(row)

df_results = pd.DataFrame(result_rows)
df_results.to_csv(str(OUT_DIR / 'predictions_step1.csv'), index=False)

# Summary metrics CSV
summary = {
    'Sharpe':           sharpe(pnl),
    'Sortino':          sortino(pnl),
    'MaxDrawdown':      max_dd,
    'HitRate':          hit_rate,
    'ProfitFactor':     profit_factor,
    'SignalCoverage':   signal_coverage,
    'PICP_80':          picp_80,
    'Calib_0.5':        calib_50,
    'QCR':              qcr,
}
pd.Series(summary).to_csv(str(OUT_DIR / 'summary_metrics.csv'), header=['value'])
print('Results saved to', OUT_DIR)